In [119]:
import os
# from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables from the .env file (if needed for other settings)
# load_dotenv(".env")

# Initialize the chat model using local Ollama
llm = init_chat_model(
    model="gemma4:31b-cloud",       # Name of the model pulled/hosted in Ollama
    model_provider="ollama",        # Specifies the Ollama provider 
    temperature=0.3,                # Controls randomness (lower means more focused or to the point)
    max_tokens=1000,                # Limits the maximum number of tokens in the response
)

# Define the system instruction to set assistant behavior
System = "you are a helpful Assistant"

# Initialize conversation history with the system instruction
messages = [
    {"role": "system", "content": System}
]

print("Chat started. Type 'quit' or 'exit' to end.\n")

# Start interactive continuous chat loop
while True:
    # Prompt the user for input
    user_input = input("You: ")
    
    if user_input.strip().lower() in ["quit", "exit"]:
        print("Goodbye!")
        break

    if not user_input.strip():
        continue

    messages.append({"role": "user", "content": user_input})

    print("AI: ", end="", flush=True)

    full_response = ""

    for chunk in llm.stream(messages):
        token = chunk.content
        full_response += token
        
        print(token, end="", flush=True)

    print("\n")

    messages.append({"role": "assistant", "content": full_response})

Chat started. Type 'quit' or 'exit' to end.

AI: Hello! How can I help you today?

Goodbye!


In [121]:
import os
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# 1. Define schema
class ChatResponse(BaseModel):
    answer: str = Field(description="Direct, helpful answer to the user's query")
    key_points: list[str] = Field(description="2-4 key takeaways or bullet points")
    sentiment: str = Field(description="Detected tone/intent: e.g. inquiry, troubleshooting, casual")

# 2. Initialize model with temperature=0
llm = init_chat_model(
    model="gemma4:31b-cloud",       # or "llama3.2"
    model_provider="ollama",        
    temperature=0.0,
    max_tokens=1000,
)

# 3. Use method="json_mode" or method="json_schema"
structured_llm = llm.with_structured_output(ChatResponse, method="json_mode")

# 4. Strongly instruct the model in the system prompt to return ONLY JSON
system_prompt = (
    "You are a helpful assistant. "
    "You MUST respond ONLY with valid JSON conforming to the requested schema. "
    "Do NOT include any introductory or conversational text outside the JSON."
)

messages = [
    {"role": "system", "content": system_prompt}
]

print("Chat started. Type 'quit' or 'exit' to end.\n")

while True:
    user_input = input("You: ")
    
    if user_input.strip().lower() in ["quit", "exit"]:
        print("Goodbye!")
        break

    if not user_input.strip():
        continue

    messages.append({"role": "user", "content": user_input})

    try:
        structured_data: ChatResponse = structured_llm.invoke(messages)

        print(f"\n[Tone/Sentiment]: {structured_data.sentiment}")
        print(f"AI: {structured_data.answer}\n")
        print("Key Points:")
        for point in structured_data.key_points:
            print(f" - {point}")
        print("-" * 40 + "\n")

        # Save model output back to history
        messages.append({
            "role": "assistant",
            "content": structured_data.model_dump_json()
        })

    except Exception as e:
        print(f"Error parsing structured response: {e}")

Chat started. Type 'quit' or 'exit' to end.



Error parsing structured response: Failed to parse ChatResponse from completion {"message": "Hello! How can I help you today?"}. Got: 3 validation errors for ChatResponse
answer
  Field required [type=missing, input_value={'message': 'Hello! How can I help you today?'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
key_points
  Field required [type=missing, input_value={'message': 'Hello! How can I help you today?'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
sentiment
  Field required [type=missing, input_value={'message': 'Hello! How can I help you today?'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
Error parsing structured response: Failed to parse ChatResponse from completion {"message": "Hello! How can I help you today?"}. Got: